# 점핏 크롤링 분석
Playwright로 실제 브라우저처럼 접근 → 네트워크 요청 가로채기 → 내부 API 발견

In [ ]:
import asyncio
import json
from playwright.async_api import async_playwright

# 점핏: 신입/경력무관 IT 직군 공고
# URL 패턴: https://jumpit.saramin.co.kr/positions?jobCategory=1&career=1
# jobCategory=1 → 개발, career=1 → 신입, career=2 → 1~3년, 0 → 경력무관?
TARGET_URL = "https://jumpit.saramin.co.kr/positions"

captured_requests = []
captured_responses = []

async def analyze_jumpit():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        ctx = await browser.new_context(
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1280, "height": 800},
            locale="ko-KR",
        )
        page = await ctx.new_page()
        
        # 네트워크 요청 가로채기
        async def on_request(request):
            url = request.url
            # API 관련 요청만 캡처
            if any(k in url for k in ['/api/', 'graphql', '.json', 'position', 'job']):
                captured_requests.append({
                    'method': request.method,
                    'url': url,
                    'headers': dict(request.headers),
                    'post_data': request.post_data,
                })
        
        async def on_response(response):
            url = response.url
            if any(k in url for k in ['/api/', 'graphql', '.json', 'position', 'job']):
                try:
                    body = await response.json()
                    captured_responses.append({
                        'url': url,
                        'status': response.status,
                        'body': body,
                    })
                except:
                    pass
        
        page.on('request', on_request)
        page.on('response', on_response)
        
        print("페이지 로드 중...")
        await page.goto(TARGET_URL, wait_until='networkidle', timeout=30000)
        await asyncio.sleep(3)  # 동적 컨텐츠 로드 대기
        
        print(f"\n캡처된 API 요청 수: {len(captured_requests)}")
        for req in captured_requests:
            print(f"  [{req['method']}] {req['url'][:100]}")
        
        print(f"\n캡처된 API 응답 수: {len(captured_responses)}")
        for resp in captured_responses:
            body_preview = str(resp['body'])[:200]
            print(f"  [{resp['status']}] {resp['url'][:80]}")
            print(f"    → {body_preview}")
        
        # 페이지 HTML 저장
        html = await page.content()
        with open('/tmp/jumpit_page.html', 'w') as f:
            f.write(html)
        print(f"\nHTML 저장됨: {len(html)} chars")
        
        await browser.close()
    
    return captured_requests, captured_responses

requests, responses = await analyze_jumpit()
print("\n분석 완료")

In [ ]:
# 응답 상세 분석
for resp in responses:
    print(f"\n=== {resp['url']} ===")
    body = resp['body']
    if isinstance(body, dict):
        print(f"최상위 키: {list(body.keys())}")
        # 공고 리스트 위치 탐색
        for key in body:
            val = body[key]
            if isinstance(val, list) and len(val) > 0:
                print(f"  [{key}] 리스트 {len(val)}건")
                if isinstance(val[0], dict):
                    print(f"    첫 항목 키: {list(val[0].keys())}")
                    print(f"    샘플: {json.dumps(val[0], ensure_ascii=False, default=str)[:300]}")
            elif isinstance(val, dict):
                print(f"  [{key}] dict: {list(val.keys())}")

In [ ]:
# HTML 파싱으로 공고 구조 분석
from bs4 import BeautifulSoup

with open('/tmp/jumpit_page.html') as f:
    html = f.read()

soup = BeautifulSoup(html, 'html.parser')

# 공고 카드 찾기 (다양한 셀렉터 시도)
selectors_to_try = [
    'a[href*="/position/"]',
    '[class*="position"]',
    '[class*="job"]',
    '[class*="card"]',
    'li[class*="item"]',
]

for sel in selectors_to_try:
    items = soup.select(sel)
    if items:
        print(f"'{sel}' → {len(items)}개")
        if items[0]:
            print(f"  첫 항목 텍스트: {items[0].get_text(strip=True)[:100]}")
            print(f"  href: {items[0].get('href', '')}")

In [ ]:
# JSON-LD 또는 __NEXT_DATA__ 에서 초기 데이터 추출
import re

# Next.js 초기 데이터
next_data_match = re.search(r'<script id="__NEXT_DATA__" type="application/json">(.*?)</script>', html, re.DOTALL)
if next_data_match:
    next_data = json.loads(next_data_match.group(1))
    print("__NEXT_DATA__ 발견!")
    print(f"최상위 키: {list(next_data.keys())}")
    props = next_data.get('props', {})
    page_props = props.get('pageProps', {})
    print(f"pageProps 키: {list(page_props.keys())}")
    print(json.dumps(page_props, ensure_ascii=False, default=str)[:2000])
else:
    print("__NEXT_DATA__ 없음 (RSC 방식)")

# React Query 캐시 데이터
query_cache = re.findall(r'window\.__REACT_QUERY_STATE__\s*=\s*(\{.*?\});', html, re.DOTALL)
if query_cache:
    print(f"\nReact Query 캐시 발견: {len(query_cache)}개")

In [ ]:
# 신입/IT 필터가 적용된 URL 테스트
# 점핏 필터 파라미터 추측:
# - jobCategory: IT 직군 카테고리 ID
# - career: 경력 조건 (0=신입, 1=경력무관?)
# - sort: 정렬

# 실제 브라우저 요청 재현
import httpx

async def test_jumpit_api(url, params=None, headers=None):
    base_headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'application/json, text/plain, */*',
        'Referer': 'https://jumpit.saramin.co.kr/positions',
        'Origin': 'https://jumpit.saramin.co.kr',
    }
    if headers:
        base_headers.update(headers)
    
    async with httpx.AsyncClient(follow_redirects=False, timeout=10) as client:
        r = await client.get(url, params=params, headers=base_headers)
        print(f"HTTP {r.status_code} | Content-Type: {r.headers.get('content-type', '')}")
        if r.status_code == 200 and 'json' in r.headers.get('content-type', ''):
            return r.json()
        elif r.status_code in (301, 302, 307, 308):
            print(f"Redirect → {r.headers.get('location', '')}")
        else:
            print(f"Response: {r.text[:300]}")
        return None

# 캡처된 요청 중 API URL들 직접 재시도
for req in requests[:5]:
    print(f"\n테스트: {req['url'][:80]}")
    await test_jumpit_api(req['url'])

In [ ]:
# Playwright로 필터 적용 후 데이터 추출
# 점핏 카테고리 탐색 + 신입 필터

job_data = []
api_calls = []

async def scrape_jumpit_with_filter():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        ctx = await browser.new_context(
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            locale="ko-KR",
        )
        page = await ctx.new_page()
        
        # 응답 가로채기
        async def on_response(response):
            url = response.url
            if 'position' in url and 'api' in url and response.status == 200:
                try:
                    ct = response.headers.get('content-type', '')
                    if 'json' in ct:
                        body = await response.json()
                        api_calls.append({'url': url, 'body': body})
                        print(f"API 캡처: {url[:80]}")
                except:
                    pass
        
        page.on('response', on_response)
        
        # jobCategory=1(개발) 필터로 이동
        await page.goto('https://jumpit.saramin.co.kr/positions?jobCategory=1', 
                        wait_until='networkidle', timeout=30000)
        await asyncio.sleep(3)
        
        # 렌더링된 HTML에서 공고 추출
        html = await page.content()
        
        # 공고 링크 추출
        links = await page.query_selector_all('a[href*="/position/"]')
        print(f"\n공고 링크 수: {len(links)}")
        
        for link in links[:5]:
            href = await link.get_attribute('href')
            text = await link.inner_text()
            print(f"  {href} | {text[:60]}")
        
        await browser.close()
    
    return html, api_calls

html, api_calls = await scrape_jumpit_with_filter()
print(f"\n캡처된 API 콜: {len(api_calls)}")

In [ ]:
# 캡처된 API 응답 분석
for call in api_calls:
    print(f"\n=== {call['url']} ===")
    body = call['body']
    print(f"타입: {type(body).__name__}")
    if isinstance(body, dict):
        print(f"키: {list(body.keys())}")
        # 공고 리스트 탐색
        def find_lists(d, path=''):
            if isinstance(d, list) and len(d) > 0 and isinstance(d[0], dict):
                print(f"  리스트 발견 [{path}]: {len(d)}건, 키={list(d[0].keys())}")
                print(f"    샘플: {json.dumps(d[0], ensure_ascii=False, default=str)[:400]}")
            elif isinstance(d, dict):
                for k, v in d.items():
                    find_lists(v, f"{path}.{k}" if path else k)
        find_lists(body)

In [ ]:
# HTML 파싱 - 공고 카드 구조 파악
soup = BeautifulSoup(html, 'html.parser')

# 공고 카드 셀렉터 탐색
print("=== 공고 카드 구조 분석 ===")
position_links = soup.select('a[href*="/position/"]')
print(f"공고 링크: {len(position_links)}개")

if position_links:
    card = position_links[0]
    # 부모 요소로 올라가서 카드 전체 구조 파악
    parent = card.parent
    while parent and parent.name not in ('li', 'article', 'div'):
        parent = parent.parent
    
    if parent:
        print(f"카드 태그: {parent.name}")
        print(f"카드 class: {parent.get('class', [])}")
        print(f"카드 내용:\n{parent.get_text(separator='|', strip=True)[:300]}")
        print(f"\n카드 HTML 구조:\n{str(parent)[:1000]}")